# Gradient Conflict Analysis for HiP-AD Multi-Task Learning

This notebook analyzes gradient conflicts between tasks (det, map, plan, ego, motion)
by computing per-task gradients on shared parameters and measuring cosine similarity.

**Usage:**
1. Set `CONFIG_PATH` to your training config
2. Set `CHECKPOINT_PATH = None` for random init, or path to a checkpoint
3. Set `NUM_BATCHES` to the number of samples to analyze
4. Run all cells

In [1]:
import os
import sys
import importlib
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import proj3d
from sklearn.decomposition import PCA
from collections import defaultdict
from itertools import combinations

matplotlib.rcParams['figure.figsize'] = (12, 8)
matplotlib.rcParams['font.size'] = 12

# Project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print(f"Working directory: {os.getcwd()}")

Working directory: /home/kyungmin/min_ws/rideflux/HiP-AD


In [2]:
# ============================================================
# Configuration - EDIT THESE
# ============================================================
CONFIG_PATH = "projects/configs/hipad_nusc_stage2.py"

# Checkpoint: None = random init, or path to a checkpoint
CHECKPOINT_PATH = None  # Start from scratch (random init)
# CHECKPOINT_PATH = "./work_dirs/hipad_nusc_stage2/latest.pth"

NUM_BATCHES = 100      # Number of batches to analyze
DEVICE = "cuda:0"      # GPU device

## 1. Load Model and Dataset

In [3]:
from mmcv import Config
from mmcv.runner import load_checkpoint
from mmcv.parallel import MMDataParallel
from mmdet.models import build_detector
from mmdet.datasets import build_dataset

# Import plugin modules
plg_lib = importlib.import_module("projects.mmdet3d_plugin")

cfg = Config.fromfile(CONFIG_PATH)

# Parse GPU index from DEVICE string
gpu_id = int(DEVICE.split(":")[-1]) if ":" in DEVICE else 0

# Build model
model = build_detector(cfg.model)

# Optionally load checkpoint, otherwise use random init
if CHECKPOINT_PATH is not None:
    checkpoint = load_checkpoint(model, CHECKPOINT_PATH, map_location="cpu")
    print(f"Model loaded from {CHECKPOINT_PATH}")
else:
    model.init_weights()
    print("Model initialized with random weights (no checkpoint)")

model = MMDataParallel(model.cuda(gpu_id), device_ids=[gpu_id])
model.train()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}, Trainable: {trainable_params:,}")

/home/kyungmin/anaconda3/envs/hipad/lib/python3.8/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(


Use flash_attn_varlen_kvpacked_func


/home/kyungmin/anaconda3/envs/hipad/lib/python3.8/site-packages/mmdet/models/backbones/resnet.py:401: UserWarning: DeprecationWarning: pretrained is deprecated, please use "init_cfg" instead
  warnings.warn('DeprecationWarning: pretrained is deprecated, '
2026-03-19 21:20:21,666 - mmcv - INFO - initialize ResNet with init_cfg {'type': 'Pretrained', 'checkpoint': 'ckpts/resnet50-19c8e357.pth'}
2026-03-19 21:20:21,667 - mmcv - INFO - load model from: ckpts/resnet50-19c8e357.pth
2026-03-19 21:20:21,669 - mmcv - INFO - load checkpoint from local path: ckpts/resnet50-19c8e357.pth
2026-03-19 21:20:21,889 - mmcv - WARNING - The model and loaded state dict do not match exactly

unexpected key in source state_dict: fc.weight, fc.bias

2026-03-19 21:20:21,904 - mmcv - INFO - initialize FPN with init_cfg {'type': 'Xavier', 'layer': 'Conv2d', 'distribution': 'uniform'}
2026-03-19 21:20:22,087 - mmcv - INFO - 
img_backbone.conv1.weight - torch.Size([64, 3, 7, 7]): 
PretrainedInit: load from ckpts/r

Model initialized with random weights (no checkpoint)
Total params: 88,797,175, Trainable: 88,432,865


In [4]:
# Build dataset and dataloader (train set for gradient analysis)
from projects.mmdet3d_plugin.datasets.builder import build_dataloader

dataset = build_dataset(cfg.data.train)
dataloader = build_dataloader(
    dataset,
    samples_per_gpu=1,  # Use batch_size=1 to save memory with retain_graph
    workers_per_gpu=2,
    dist=False,
    shuffle=True,
)
data_iter = iter(dataloader)
print(f"Dataset size: {len(dataset)}, Dataloader batches: {len(dataloader)}")

{'version': 'v1.0-trainval'}
WARNING!!!!, Only can be used for obtain inference speed!!!!
Dataset size: 28130, Dataloader batches: 28130


## 2. Define Shared Parameters and Task Loss Grouping

In [5]:
# Identify shared vs task-specific parameters
# Shared: backbone, neck, decoder attention/FFN layers
# Task-specific: refine layers, task-specific samplers
# NOTE: Use model.module to access actual parameters (model is wrapped in MMDataParallel)

TASK_SPECIFIC_KEYWORDS = [
    "det_refine", "map_refine", "ego_refine", "plan_refine", "motion_refine",
    "det_sampler", "map_sampler", "plan_sampler", "motion_sampler", "align_sampler",
    "det_instance_bank", "map_instance_bank", "ego_instance_bank", "plan_instance_bank",
    "det_anchor_encoder", "map_anchor_encoder", "plan_anchor_encoder",
    "loss_det", "loss_map", "loss_ego", "loss_plan", "loss_motion",
]

def is_shared_param(name):
    """Check if a parameter is shared across tasks."""
    return not any(kw in name for kw in TASK_SPECIFIC_KEYWORDS)

shared_params = {name: p for name, p in model.module.named_parameters()
                 if p.requires_grad and is_shared_param(name)}
task_specific_params = {name: p for name, p in model.module.named_parameters()
                        if p.requires_grad and not is_shared_param(name)}

shared_numel = sum(p.numel() for p in shared_params.values())
task_numel = sum(p.numel() for p in task_specific_params.values())

print(f"Shared parameters: {len(shared_params)} tensors, {shared_numel:,} params")
print(f"Task-specific parameters: {len(task_specific_params)} tensors, {task_numel:,} params")
print(f"\nShared param groups (first 10):")
for i, name in enumerate(list(shared_params.keys())[:10]):
    print(f"  {name}: {shared_params[name].shape}")
print("  ...")

Shared parameters: 741 tensors, 75,330,255 params
Task-specific parameters: 738 tensors, 13,102,610 params

Shared param groups (first 10):
  img_backbone.conv1.weight: torch.Size([64, 3, 7, 7])
  img_backbone.bn1.weight: torch.Size([64])
  img_backbone.bn1.bias: torch.Size([64])
  img_backbone.layer1.0.conv1.weight: torch.Size([64, 64, 1, 1])
  img_backbone.layer1.0.bn1.weight: torch.Size([64])
  img_backbone.layer1.0.bn1.bias: torch.Size([64])
  img_backbone.layer1.0.conv2.weight: torch.Size([64, 64, 3, 3])
  img_backbone.layer1.0.bn2.weight: torch.Size([64])
  img_backbone.layer1.0.bn2.bias: torch.Size([64])
  img_backbone.layer1.0.conv3.weight: torch.Size([256, 64, 1, 1])
  ...


In [6]:
# Define how to group loss keys by task
TASK_NAMES = ["det", "map", "ego", "plan", "motion"]

def group_losses_by_task(loss_dict):
    """Group loss dict keys by task and sum per-task losses."""
    task_losses = {}
    for task in TASK_NAMES:
        task_loss_keys = [k for k in loss_dict if k.startswith(f"{task}_loss")]
        if task_loss_keys:
            task_losses[task] = sum(loss_dict[k] for k in task_loss_keys)
    # depth loss is shared auxiliary - exclude from conflict analysis
    return task_losses

print("Task loss grouping defined for:", TASK_NAMES)

Task loss grouping defined for: ['det', 'map', 'ego', 'plan', 'motion']


## 3. Compute Per-Task Gradients

In [7]:
def get_task_gradients(model, data_batch, shared_params, device):
    """
    Compute per-task gradients on shared parameters.
    Returns gradients moved to CPU to avoid GPU OOM.
    """
    with torch.cuda.amp.autocast(enabled=False):
        loss_dict = model(**data_batch)
    
    task_losses = group_losses_by_task(loss_dict)
    
    task_grads = {}
    param_list = list(shared_params.values())
    
    for task_name, task_loss in task_losses.items():
        model.zero_grad()
        task_loss.backward(retain_graph=True)
        
        grad_vec = []
        for p in param_list:
            if p.grad is not None:
                grad_vec.append(p.grad.detach().flatten())
            else:
                grad_vec.append(torch.zeros(p.numel(), device=device))
        # Move to CPU immediately to free GPU memory
        task_grads[task_name] = torch.cat(grad_vec).cpu()
    
    model.zero_grad()
    
    task_loss_values = {k: v.item() for k, v in task_losses.items()}
    return task_grads, task_loss_values


def compute_cosine_similarity_matrix(task_grads):
    """Compute pairwise cosine similarity between task gradients (on CPU)."""
    tasks = sorted(task_grads.keys())
    n = len(tasks)
    cos_sim = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            g_i = task_grads[tasks[i]].float()
            g_j = task_grads[tasks[j]].float()
            cos_sim[i, j] = F.cosine_similarity(
                g_i.unsqueeze(0), g_j.unsqueeze(0)
            ).item()
    
    return cos_sim, tasks

print("Gradient extraction & cosine similarity functions defined.")

Gradient extraction & cosine similarity functions defined.


In [8]:
# Run gradient analysis over multiple batches
# Stats are computed on-the-fly to avoid storing all gradient vectors (would be ~150GB for 100 batches)

PCA_SAMPLE_BATCHES = 5  # Only keep raw gradients for this many batches (for PCA visualization)

all_cos_sims = []           # Per-batch cosine similarity matrices
all_task_losses = []        # Per-batch loss values
all_grad_norms = defaultdict(list)  # Per-batch gradient norms
pca_task_grads = []         # Raw gradients for PCA (only first few batches)

print(f"Analyzing {NUM_BATCHES} batches (keeping {PCA_SAMPLE_BATCHES} for PCA)...")
for batch_idx in range(NUM_BATCHES):
    try:
        data_batch = next(data_iter)
    except StopIteration:
        data_iter = iter(dataloader)
        data_batch = next(data_iter)
    
    task_grads, task_loss_values = get_task_gradients(
        model, data_batch, shared_params, DEVICE
    )
    
    # Compute cosine similarity on-the-fly
    cos_sim, tasks = compute_cosine_similarity_matrix(task_grads)
    all_cos_sims.append(cos_sim)
    all_task_losses.append(task_loss_values)
    
    # Store gradient norms
    for task, grad in task_grads.items():
        all_grad_norms[task].append(grad.float().norm().item())
    
    # Keep raw gradients only for PCA samples
    if batch_idx < PCA_SAMPLE_BATCHES:
        pca_task_grads.append(task_grads)
    
    # Print progress
    if batch_idx % 10 == 0 or batch_idx < 5:
        tasks_str = ", ".join(f"{k}: {v:.4f}" for k, v in task_loss_values.items())
        print(f"  Batch {batch_idx}: {tasks_str}")
    
    del task_grads
    torch.cuda.empty_cache()

print(f"Done! Collected {len(all_cos_sims)} cosine similarity matrices.")

Analyzing 100 batches (keeping 5 for PCA)...


OutOfMemoryError: CUDA out of memory. Tried to allocate 66.00 MiB (GPU 0; 23.65 GiB total capacity; 3.17 GiB already allocated; 10.25 MiB free; 3.38 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

## 4. Cosine Similarity Heatmap (Gradient Conflict Detection)

In [ ]:
# Cosine similarity is already computed in the main loop
avg_cos_sim = np.mean(all_cos_sims, axis=0)
std_cos_sim = np.std(all_cos_sims, axis=0)

# Plot heatmap
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

im1 = axes[0].imshow(avg_cos_sim, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[0].set_xticks(range(len(tasks)))
axes[0].set_yticks(range(len(tasks)))
axes[0].set_xticklabels(tasks, fontsize=14)
axes[0].set_yticklabels(tasks, fontsize=14)
axes[0].set_title(f'Average Gradient Cosine Similarity\n({NUM_BATCHES} batches)', fontsize=14)
for i in range(len(tasks)):
    for j in range(len(tasks)):
        color = 'white' if abs(avg_cos_sim[i, j]) > 0.5 else 'black'
        axes[0].text(j, i, f'{avg_cos_sim[i, j]:.3f}', ha='center', va='center',
                     fontsize=12, fontweight='bold', color=color)
plt.colorbar(im1, ax=axes[0], shrink=0.8)

im2 = axes[1].imshow(std_cos_sim, cmap='Oranges', vmin=0, aspect='auto')
axes[1].set_xticks(range(len(tasks)))
axes[1].set_yticks(range(len(tasks)))
axes[1].set_xticklabels(tasks, fontsize=14)
axes[1].set_yticklabels(tasks, fontsize=14)
axes[1].set_title(f'Std Dev of Cosine Similarity\n({NUM_BATCHES} batches)', fontsize=14)
for i in range(len(tasks)):
    for j in range(len(tasks)):
        axes[1].text(j, i, f'{std_cos_sim[i, j]:.3f}', ha='center', va='center',
                     fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.savefig('gradient_conflict_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== Gradient Conflict Summary ===")
print("Negative cosine similarity = CONFLICT (gradients point in opposite directions)")
print("Positive cosine similarity = ALIGNED (gradients point in similar directions)\n")
for i, j in combinations(range(len(tasks)), 2):
    sim = avg_cos_sim[i, j]
    status = "CONFLICT" if sim < 0 else "ALIGNED" if sim > 0.1 else "NEAR-ORTHOGONAL"
    print(f"  {tasks[i]:8s} vs {tasks[j]:8s}: {sim:+.4f}  [{status}]")

## 5. Per-Batch Cosine Similarity Trend

In [ ]:
# Plot per-batch cosine similarity trend (line plot, no markers for readability)
task_pairs = list(combinations(range(len(tasks)), 2))
n_pairs = len(task_pairs)

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.tab10(np.linspace(0, 1, n_pairs))

for idx, (i, j) in enumerate(task_pairs):
    values = [cs[i, j] for cs in all_cos_sims]
    label = f"{tasks[i]} vs {tasks[j]}"
    ax.plot(range(NUM_BATCHES), values, '-', color=colors[idx], label=label, linewidth=1.5, alpha=0.8)

ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='conflict threshold')
ax.set_xlabel('Batch Index', fontsize=13)
ax.set_ylabel('Cosine Similarity', fontsize=13)
ax.set_title('Per-Batch Gradient Cosine Similarity Between Task Pairs', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-1.1, 1.1)

plt.tight_layout()
plt.savefig('gradient_conflict_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Gradient Magnitude Analysis

In [ ]:
# Gradient norms already collected in main loop (all_grad_norms)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

avg_norms = {k: np.mean(v) for k, v in all_grad_norms.items()}
std_norms = {k: np.std(v) for k, v in all_grad_norms.items()}
task_names = sorted(avg_norms.keys())
x = range(len(task_names))

bars = axes[0].bar(x, [avg_norms[t] for t in task_names],
                   yerr=[std_norms[t] for t in task_names],
                   capsize=5, color=plt.cm.Set2(np.linspace(0, 1, len(task_names))))
axes[0].set_xticks(x)
axes[0].set_xticklabels(task_names, fontsize=13)
axes[0].set_ylabel('Gradient L2 Norm', fontsize=13)
axes[0].set_title(f'Average Gradient Magnitude per Task\n(on shared parameters, {NUM_BATCHES} batches)', fontsize=14)
axes[0].grid(True, alpha=0.3, axis='y')

total_norm = sum(avg_norms.values())
rel_norms = {k: v / total_norm * 100 for k, v in avg_norms.items()}
axes[1].pie([rel_norms[t] for t in task_names], labels=task_names,
            autopct='%1.1f%%', startangle=90,
            colors=plt.cm.Set2(np.linspace(0, 1, len(task_names))))
axes[1].set_title('Relative Gradient Magnitude Share', fontsize=14)

plt.tight_layout()
plt.savefig('gradient_magnitude.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== Gradient Magnitude Summary ===")
print("Large magnitude imbalance can also cause optimization issues.\n")
for t in task_names:
    print(f"  {t:8s}: norm={avg_norms[t]:.4f} (+/- {std_norms[t]:.4f}), share={rel_norms[t]:.1f}%")

## 7. PCA Gradient Direction Visualization (2D & 3D)

In [ ]:
# 3D arrow patch for matplotlib
class Arrow3D(FancyArrowPatch):
    def __init__(self, xs, ys, zs, *args, **kwargs):
        super().__init__((0, 0), (0, 0), *args, **kwargs)
        self._verts3d = xs, ys, zs

    def do_3d_projection(self, renderer=None):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, self.axes.M)
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))
        return min(zs)


def visualize_gradients_pca(pca_task_grads, batch_idx=0):
    """Visualize task gradient directions using PCA projection."""
    task_grads = pca_task_grads[batch_idx]
    tasks = sorted(task_grads.keys())
    
    grad_matrix = torch.stack([task_grads[t].float() for t in tasks]).numpy()
    
    pca = PCA(n_components=3)
    grad_3d = pca.fit_transform(grad_matrix)
    
    norms = np.linalg.norm(grad_3d, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-8, None)
    grad_3d_unit = grad_3d / norms
    rel_norms_pca = norms / norms.max()
    grad_3d_scaled = grad_3d_unit * rel_norms_pca
    
    task_colors = {
        'det': '#e74c3c', 'map': '#2ecc71', 'plan': '#3498db',
        'ego': '#f39c12', 'motion': '#9b59b6',
    }
    
    fig = plt.figure(figsize=(18, 7))
    
    # --- 2D ---
    ax1 = fig.add_subplot(121)
    for i, task in enumerate(tasks):
        ax1.annotate('', xy=(grad_3d_scaled[i, 0], grad_3d_scaled[i, 1]),
                     xytext=(0, 0),
                     arrowprops=dict(arrowstyle='->', color=task_colors.get(task, 'gray'),
                                     lw=3, mutation_scale=20))
        ax1.text(grad_3d_scaled[i, 0] * 1.15, grad_3d_scaled[i, 1] * 1.15,
                 task, fontsize=14, fontweight='bold',
                 color=task_colors.get(task, 'gray'), ha='center')
    
    cos_sim, _ = compute_cosine_similarity_matrix(task_grads)
    for (i, j) in combinations(range(len(tasks)), 2):
        if cos_sim[i, j] < 0:
            mid_x = (grad_3d_scaled[i, 0] + grad_3d_scaled[j, 0]) / 2
            mid_y = (grad_3d_scaled[i, 1] + grad_3d_scaled[j, 1]) / 2
            angle = np.degrees(np.arccos(np.clip(cos_sim[i, j], -1, 1)))
            ax1.annotate(f'{angle:.0f}deg', xy=(mid_x, mid_y),
                        fontsize=9, color='red', alpha=0.8,
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='yellow', alpha=0.5))
    
    lim = 1.4
    ax1.set_xlim(-lim, lim); ax1.set_ylim(-lim, lim)
    ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=13)
    ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=13)
    ax1.set_title(f'Task Gradient Directions (PCA 2D)\nBatch {batch_idx}', fontsize=14)
    ax1.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
    ax1.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
    ax1.set_aspect('equal'); ax1.grid(True, alpha=0.2)
    
    # --- 3D ---
    ax2 = fig.add_subplot(122, projection='3d')
    for i, task in enumerate(tasks):
        color = task_colors.get(task, 'gray')
        arrow = Arrow3D([0, grad_3d_scaled[i, 0]], [0, grad_3d_scaled[i, 1]],
                        [0, grad_3d_scaled[i, 2]], mutation_scale=20, lw=3, arrowstyle='->', color=color)
        ax2.add_artist(arrow)
        ax2.text(grad_3d_scaled[i, 0]*1.2, grad_3d_scaled[i, 1]*1.2,
                 grad_3d_scaled[i, 2]*1.2, task, fontsize=13, fontweight='bold', color=color)
    
    ax2.set_xlim(-lim, lim); ax2.set_ylim(-lim, lim); ax2.set_zlim(-lim, lim)
    ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=11)
    ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=11)
    ax2.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.1f}%)', fontsize=11)
    ax2.set_title(f'Task Gradient Directions (PCA 3D)\nBatch {batch_idx}', fontsize=14)
    
    plt.tight_layout()
    plt.savefig(f'gradient_directions_batch{batch_idx}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"PCA explained variance: {pca.explained_variance_ratio_ * 100}")
    print(f"Total explained: {sum(pca.explained_variance_ratio_) * 100:.1f}%")

visualize_gradients_pca(pca_task_grads, batch_idx=0)

In [ ]:
# Visualize PCA sample batches overlaid (2D, using only the saved pca_task_grads)
fig, ax = plt.subplots(figsize=(10, 10))

task_colors = {
    'det': '#e74c3c', 'map': '#2ecc71', 'plan': '#3498db',
    'ego': '#f39c12', 'motion': '#9b59b6',
}

all_grads_list = []
all_labels = []
for batch_idx, task_grads in enumerate(pca_task_grads):
    tasks_sorted = sorted(task_grads.keys())
    for t in tasks_sorted:
        all_grads_list.append(task_grads[t].float().numpy())
        all_labels.append((t, batch_idx))

all_grads_matrix = np.stack(all_grads_list)
pca = PCA(n_components=2)
all_grads_2d = pca.fit_transform(all_grads_matrix)

norms = np.linalg.norm(all_grads_2d, axis=1, keepdims=True)
norms = np.clip(norms, 1e-8, None)
all_grads_2d_scaled = all_grads_2d / norms.max() * 0.9

for idx, (label, batch_idx) in enumerate(all_labels):
    alpha = 0.3 + 0.7 * (batch_idx / max(PCA_SAMPLE_BATCHES - 1, 1))
    color = task_colors.get(label, 'gray')
    ax.annotate('', xy=(all_grads_2d_scaled[idx, 0], all_grads_2d_scaled[idx, 1]),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2, alpha=alpha,
                                mutation_scale=15))

for task, color in task_colors.items():
    ax.plot([], [], '-', color=color, lw=3, label=task)
ax.legend(fontsize=13, loc='upper right')

lim = 1.2
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=13)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=13)
ax.set_title(f'Gradient Directions ({PCA_SAMPLE_BATCHES} batches, PCA 2D)\n'
             f'Lighter=earlier batch, Darker=later batch', fontsize=14)
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
ax.set_aspect('equal'); ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('gradient_directions_all_batches.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Per-Layer Gradient Conflict Analysis

Analyze which layers (backbone, neck, decoder) have the most gradient conflict.

In [ ]:
def compute_layer_group_conflicts(model, data_batch, device):
    """Compute gradient cosine similarity per parameter group."""
    
    # Define parameter groups (use model.module for MMDataParallel)
    param_groups = {
        'backbone': {},
        'neck': {},
        'decoder_attn': {},
        'decoder_ffn': {},
        'decoder_other': {},
    }
    
    for name, p in model.module.named_parameters():
        if not p.requires_grad or not is_shared_param(name):
            continue
        if 'img_backbone' in name:
            param_groups['backbone'][name] = p
        elif 'img_neck' in name:
            param_groups['neck'][name] = p
        elif 'graph_model' in name or 'temp_graph' in name or 'inter_graph' in name:
            param_groups['decoder_attn'][name] = p
        elif 'ffn' in name:
            param_groups['decoder_ffn'][name] = p
        else:
            param_groups['decoder_other'][name] = p
    
    # MMDataParallel handles DataContainer scattering
    with torch.cuda.amp.autocast(enabled=False):
        loss_dict = model(**data_batch)
    
    task_losses = group_losses_by_task(loss_dict)
    tasks = sorted(task_losses.keys())
    
    # Compute per-group, per-task gradients
    group_task_grads = {}
    for group_name, params in param_groups.items():
        if not params:
            continue
        group_task_grads[group_name] = {}
        param_list = list(params.values())
        
        for task_name, task_loss in task_losses.items():
            model.zero_grad()
            task_loss.backward(retain_graph=True)
            grad_vec = []
            for p in param_list:
                if p.grad is not None:
                    grad_vec.append(p.grad.detach().flatten())
                else:
                    grad_vec.append(torch.zeros(p.numel(), device=device))
            group_task_grads[group_name][task_name] = torch.cat(grad_vec)
    
    model.zero_grad()
    
    # Compute cosine similarity per group
    group_cos_sims = {}
    for group_name, tg in group_task_grads.items():
        cos_sim, _ = compute_cosine_similarity_matrix(tg)
        group_cos_sims[group_name] = cos_sim
    
    return group_cos_sims, tasks

# Run on first batch
data_iter2 = iter(dataloader)
data_batch = next(data_iter2)
group_cos_sims, tasks = compute_layer_group_conflicts(model, data_batch, DEVICE)
torch.cuda.empty_cache()

# Plot
n_groups = len(group_cos_sims)
fig, axes = plt.subplots(1, n_groups, figsize=(5 * n_groups, 4.5))
if n_groups == 1:
    axes = [axes]

for ax, (group_name, cos_sim) in zip(axes, group_cos_sims.items()):
    im = ax.imshow(cos_sim, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(tasks)))
    ax.set_yticks(range(len(tasks)))
    ax.set_xticklabels(tasks, fontsize=11, rotation=45)
    ax.set_yticklabels(tasks, fontsize=11)
    ax.set_title(group_name, fontsize=13, fontweight='bold')
    for i in range(len(tasks)):
        for j in range(len(tasks)):
            color = 'white' if abs(cos_sim[i, j]) > 0.5 else 'black'
            ax.text(j, i, f'{cos_sim[i,j]:.2f}', ha='center', va='center',
                    fontsize=10, color=color)

plt.suptitle('Gradient Conflict by Parameter Group', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('gradient_conflict_per_layer.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
print("\n=== Per-Layer Conflict Summary ===")
for group_name, cos_sim in group_cos_sims.items():
    min_sim = np.min(cos_sim[np.triu_indices(len(tasks), k=1)])
    avg_sim = np.mean(cos_sim[np.triu_indices(len(tasks), k=1)])
    print(f"  {group_name:20s}: avg_sim={avg_sim:+.4f}, min_sim={min_sim:+.4f}")

## 9. PCGrad Simulation

Simulate what PCGrad would do: project conflicting gradients to remove the conflicting component.

In [ ]:
def pcgrad_project(task_grads):
    """
    Simulate PCGrad on CPU.
    """
    tasks = sorted(task_grads.keys())
    grads = {t: task_grads[t].float().cpu() for t in tasks}
    
    conflict_count = 0
    total_pairs = 0
    projected_grads = {}
    projection_magnitudes = defaultdict(list)
    
    for t_i in tasks:
        g_i = grads[t_i].clone()
        for t_j in tasks:
            if t_i == t_j:
                continue
            g_j = grads[t_j]
            dot = torch.dot(g_i, g_j)
            if dot < 0:
                conflict_count += 1
                proj = dot / (torch.dot(g_j, g_j) + 1e-8) * g_j
                proj_magnitude = proj.norm().item() / (g_i.norm().item() + 1e-8)
                projection_magnitudes[f"{t_i}<-{t_j}"].append(proj_magnitude)
                g_i = g_i - proj
            total_pairs += 1
        projected_grads[t_i] = g_i
    
    angle_changes = {}
    for t in tasks:
        cos = F.cosine_similarity(grads[t].unsqueeze(0), projected_grads[t].unsqueeze(0)).item()
        angle_changes[t] = np.degrees(np.arccos(np.clip(cos, -1, 1)))
    
    return {
        'projected_grads': projected_grads,
        'original_grads': grads,
        'conflict_ratio': conflict_count / max(total_pairs, 1),
        'conflict_count': conflict_count,
        'total_pairs': total_pairs,
        'angle_changes': angle_changes,
        'projection_magnitudes': projection_magnitudes,
    }

# Run PCGrad simulation on the PCA sample batches (raw gradients available)
pcgrad_results = []
for batch_idx, task_grads in enumerate(pca_task_grads):
    result = pcgrad_project(task_grads)
    pcgrad_results.append(result)

print("=== PCGrad Simulation Results ===")
print(f"(on {len(pca_task_grads)} sample batches)\n")
print(f"Conflict ratio per batch:")
for i, r in enumerate(pcgrad_results):
    print(f"  Batch {i}: {r['conflict_count']}/{r['total_pairs']} pairs conflicting "
          f"({r['conflict_ratio']*100:.1f}%)")

avg_conflict_ratio = np.mean([r['conflict_ratio'] for r in pcgrad_results])
print(f"\nAverage conflict ratio: {avg_conflict_ratio*100:.1f}%")

print(f"\nAngle change after PCGrad projection (degrees):")
for task in sorted(pcgrad_results[0]['angle_changes'].keys()):
    angles = [r['angle_changes'][task] for r in pcgrad_results]
    print(f"  {task:8s}: {np.mean(angles):.1f} +/- {np.std(angles):.1f} deg")

print(f"\nInterpretation:")
print(f"  - High conflict ratio -> PCGrad or similar method would be beneficial")
print(f"  - Large angle change -> significant gradient modification needed")
print(f"  - If conflict ratio < 20%, gradient conflict may not be the main issue")

In [ ]:
# Visualize original vs PCGrad-projected gradients
result = pcgrad_results[0]
tasks = sorted(result['original_grads'].keys())

# PCA on combined original + projected
all_vecs = []
all_info = []
for t in tasks:
    all_vecs.append(result['original_grads'][t].cpu().numpy())
    all_info.append((t, 'original'))
    all_vecs.append(result['projected_grads'][t].cpu().numpy())
    all_info.append((t, 'pcgrad'))

all_vecs = np.stack(all_vecs)
pca = PCA(n_components=2)
vecs_2d = pca.fit_transform(all_vecs)

# Normalize
max_norm = np.linalg.norm(vecs_2d, axis=1).max()
vecs_2d = vecs_2d / max_norm * 0.9

task_colors = {
    'det': '#e74c3c', 'map': '#2ecc71', 'plan': '#3498db',
    'ego': '#f39c12', 'motion': '#9b59b6',
}

fig, ax = plt.subplots(figsize=(10, 10))

for idx, (task, kind) in enumerate(all_info):
    color = task_colors.get(task, 'gray')
    linestyle = '-' if kind == 'original' else '--'
    alpha = 1.0 if kind == 'original' else 0.6
    lw = 3 if kind == 'original' else 2
    
    ax.annotate('', xy=(vecs_2d[idx, 0], vecs_2d[idx, 1]),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw,
                                alpha=alpha, linestyle=linestyle,
                                mutation_scale=18))
    if kind == 'original':
        ax.text(vecs_2d[idx, 0] * 1.12, vecs_2d[idx, 1] * 1.12,
                task, fontsize=14, fontweight='bold', color=color, ha='center')

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='gray', lw=3, label='Original gradient'),
    Line2D([0], [0], color='gray', lw=2, linestyle='--', alpha=0.6, label='After PCGrad'),
]
ax.legend(handles=legend_elements, fontsize=13, loc='upper right')

lim = 1.3
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=13)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=13)
ax.set_title('Original vs PCGrad-Projected Gradients\n'
             'Solid=Original, Dashed=After Projection', fontsize=14)
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('gradient_pcgrad_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Summary & Next Steps

In [ ]:
print("="*60)
print("GRADIENT CONFLICT ANALYSIS SUMMARY")
print("="*60)

print(f"\n1. COSINE SIMILARITY (avg over {NUM_BATCHES} batches):")
for i, j in combinations(range(len(tasks)), 2):
    sim = avg_cos_sim[i, j]
    status = "CONFLICT" if sim < 0 else "ALIGNED" if sim > 0.1 else "ORTHOGONAL"
    bar = '|' + '#' * int(abs(sim) * 20) + ' ' * (20 - int(abs(sim) * 20)) + '|'
    print(f"  {tasks[i]:8s} vs {tasks[j]:8s}: {sim:+.4f} {bar} {status}")

print(f"\n2. GRADIENT MAGNITUDE DOMINANCE:")
for t in sorted(avg_norms.keys(), key=lambda x: avg_norms[x], reverse=True):
    bar = '#' * int(rel_norms[t] / 2)
    print(f"  {t:8s}: {avg_norms[t]:.4f} ({rel_norms[t]:.1f}%) {bar}")

print(f"\n3. PCGrad CONFLICT RATIO: {avg_conflict_ratio*100:.1f}%")

print(f"\n4. RECOMMENDATIONS:")
if avg_conflict_ratio > 0.3:
    print("  [HIGH CONFLICT] Consider:")
    print("  - PCGrad / CAGrad / Nash-MTL optimizer")
    print("  - GradNorm for dynamic loss weighting")
    print("  - Task-specific learning rates")
elif avg_conflict_ratio > 0.1:
    print("  [MODERATE CONFLICT] Consider:")
    print("  - GradNorm for gradient magnitude balancing")
    print("  - Uncertainty-based loss weighting")
    print("  - Monitor if conflict increases during training")
else:
    print("  [LOW CONFLICT] Gradient conflicts are minimal.")
    print("  - Focus on loss weighting / learning rate tuning instead")
    print("  - PCGrad overhead may not be justified")

# Check gradient dominance
max_norm_task = max(avg_norms, key=avg_norms.get)
min_norm_task = min(avg_norms, key=avg_norms.get)
dominance_ratio = avg_norms[max_norm_task] / (avg_norms[min_norm_task] + 1e-8)
if dominance_ratio > 5:
    print(f"\n  [WARNING] Gradient magnitude imbalance: {max_norm_task} is {dominance_ratio:.1f}x "
          f"larger than {min_norm_task}")
    print(f"  Consider GradNorm or per-task gradient clipping")